# Eksplorasi: RAG Pipeline (Generasi Jawaban LLM)

Notebook ini menghubungkan *vector store* yang sudah kita buat sebelumnya dengan **LLM (GLM-5 Turbo dari SumoPod)**. Saat Anda bertanya, sistem akan mencari teks relevan dan memintanya untuk dirangkum oleh AI.

In [1]:
import os
import re
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

# Memuat variabel lingkungan (.env)
load_dotenv('../.env')

# 1. Inisialisasi Model LLM (Chat) dan Embedding
embeddings = OpenAIEmbeddings(
    openai_api_key=os.getenv("SUMOPOD_API_KEY"),
    openai_api_base=os.getenv("SUMOPOD_API_BASE"),
    model="text-embedding-3-small"
)

llm = ChatOpenAI(
    openai_api_key=os.getenv("SUMOPOD_API_KEY"),
    openai_api_base=os.getenv("SUMOPOD_API_BASE"),
    model="glm-5-turbo", # Model GLM lebih cepat dari SumoPod
    temperature=0.0 # Temperature 0 agar AI menjawab secara faktual, bukan kreatif/mengarang
)
print("LLM & Embedding berhasil diinisialisasi.")

LLM & Embedding berhasil diinisialisasi.


### 1. Memuat Ulang Database Vector (ChromaDB)
Kita tidak perlu lagi membaca PDF dari awal, cukup load database yang ada di folder `vector_store`.

In [2]:
persist_directory = "../vector_store"

# Memuat ChromaDB dari disk lokal
vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

# Membuat objek 'retriever' yang akan mengambil 2 teks paling mirip agar respons lebih cepat
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Database lokal berhasil dihubungkan.")

Database lokal berhasil dihubungkan.


In [3]:
# Cek jumlah halaman yang berhasil disimpan di Vector Store
all_data = vectorstore.get()
halaman_unik = set()

for metadata in all_data['metadatas']:
    if metadata and 'page' in metadata:
        halaman_unik.add(metadata['page'])

halaman_terurut = sorted(list(halaman_unik))

print(f"Total chunk (potongan teks) di database: {len(all_data['ids'])}")
print(f"Total halaman yang di-vektorisasi: {len(halaman_unik)} halaman")
if len(halaman_terurut) > 0:
    print(f"Halaman yang tercakup: {halaman_terurut[0]} sampai {halaman_terurut[-1]}")


Total chunk (potongan teks) di database: 975
Total halaman yang di-vektorisasi: 315 halaman
Halaman yang tercakup: 0 sampai 320


### 2. Menyusun Prompt (Instruksi Sistem)
Kita memberikan instruksi ketat agar LLM bertindak sebagai asisten dokumen.

In [4]:
system_prompt = (
    "Anda adalah asisten cerdas untuk tugas menjawab pertanyaan berdasarkan dokumen.\n"
    "Gunakan potongan konteks yang diambil berikut ini untuk menjawab pertanyaan.\n"
    "Jika Anda tidak tahu jawabannya, katakan saja bahwa Anda tidak tahu, jangan mencoba mengarang jawaban.\n"
    "Jaga agar jawaban tetap ringkas, padat, dan jelas (maksimal 3 paragraph 5 kalimat jika memungkinkan).\n"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

### 3. Membangun dan Menguji RAG Pipeline

In [5]:
pertanyaan = "apa isi halaman 10"
print(f"\nMemproses pertanyaan: '{pertanyaan}'...\n")

# Fast path: kalau user hanya minta isi halaman tertentu, ambil langsung dari Chroma.
# Ini tidak memanggil LLM, jadi jauh lebih cepat.
match = re.search(r"\bhal(?:aman|amaan)?\.?\s*(\d+)\b", pertanyaan.lower())

if match:
    page_number = int(match.group(1))
    page_index = page_number - 1  # metadata page dari PDF loader dimulai dari 0

    results = vectorstore.get(where={"page": page_index})
    documents = results.get("documents", [])
    text = "\n\n".join(documents)

    print(f"ISI HALAMAN {page_number}:")
    print(text[:2500])
    if len(text) > 2500:
        print("\n[Dipangkas agar output tetap ringan.]")

    print("\n> Bukti/Sumber Dokumen:")
    print(f"  - Halaman {page_number}: {len(documents)} chunk ditemukan")
else:
    # Jalur RAG normal: retrieval -> prompt -> LLM.
    docs = retriever.invoke(pertanyaan)
    context_str = "\n\n".join(doc.page_content for doc in docs)
    messages = prompt.format_messages(context=context_str, input=pertanyaan)
    response = llm.invoke(messages)

    print("==================== JAWABAN AI ====================")
    print(response.content)
    print("====================================================")
    print("\n> Bukti/Sumber Dokumen:")
    for doc in docs:
        clean_content = doc.page_content[:150].replace('\n', ' ')
        print(f"  - Halaman {doc.metadata.get('page', 'Unknown')}: {clean_content}...")


Memproses pertanyaan: 'apa isi halaman 10'...

ISI HALAMAN 10:
Kegiatan 
Belaj 
ar 
Definisi dan Makna 
| 
Kebij akan Publik 
oba Anda perhatikan tentang kehidupan kita sehari-hari, baik yang menyangkut 
kehidupan ekonomi, sosial, politik, budaya, keamanan, pertahanan, lingkungan 
hidup, dan sebagainya senantiasa terkait dengan kebijakan publik di tingkat nasional, 
provinsi, dan lokal bahkan bukannya tidak mungkin di tingkat internasional. Kita tidak 
pernah bisa lepas dari berbagai masalah kebijakan (policy issues) baik yang ringan, 
sedang, berat ataupun pada aras mikro (kecil ), meso (sedang), dan makro (besar dan 
luas ). 
Bahkan disadari atau tidak perjalanan kehidupan kita ini juga banyak dipengaruhi 
oleh adanya ‘lingkungan’ dan implementasi berbagai jenis kebijakan publik pada tingkat 
lokal, nasional, regional, dan internasional. 
Demikian besarnya pengaruh kebijakan publik dalam kehidupan kita 
maka 
tidak heran banyak pihak termasuk mahasiswa ingin mempelajari dan mengkaji

In [6]:
# Uji cepat tanpa LLM: ambil isi halaman langsung dari metadata Chroma.
pertanyaan = "apa isi halamaan 10"
print(f"\nMemproses pertanyaan: '{pertanyaan}'...\n")

match = re.search(r"\bhal(?:aman|amaan)?\.?\s*(\d+)\b", pertanyaan.lower())
page_number = int(match.group(1))
page_index = page_number - 1

results = vectorstore.get(where={"page": page_index})
documents = results.get("documents", [])
text = "\n\n".join(documents)

print(f"ISI HALAMAN {page_number}:")
print(text[:2500])
if len(text) > 2500:
    print("\n[Dipangkas agar output tetap ringan.]")

print("\n> Bukti/Sumber Dokumen:")
print(f"  - Halaman {page_number}: {len(documents)} chunk ditemukan")


Memproses pertanyaan: 'apa isi halamaan 10'...

ISI HALAMAN 10:
Kegiatan 
Belaj 
ar 
Definisi dan Makna 
| 
Kebij akan Publik 
oba Anda perhatikan tentang kehidupan kita sehari-hari, baik yang menyangkut 
kehidupan ekonomi, sosial, politik, budaya, keamanan, pertahanan, lingkungan 
hidup, dan sebagainya senantiasa terkait dengan kebijakan publik di tingkat nasional, 
provinsi, dan lokal bahkan bukannya tidak mungkin di tingkat internasional. Kita tidak 
pernah bisa lepas dari berbagai masalah kebijakan (policy issues) baik yang ringan, 
sedang, berat ataupun pada aras mikro (kecil ), meso (sedang), dan makro (besar dan 
luas ). 
Bahkan disadari atau tidak perjalanan kehidupan kita ini juga banyak dipengaruhi 
oleh adanya ‘lingkungan’ dan implementasi berbagai jenis kebijakan publik pada tingkat 
lokal, nasional, regional, dan internasional. 
Demikian besarnya pengaruh kebijakan publik dalam kehidupan kita 
maka 
tidak heran banyak pihak termasuk mahasiswa ingin mempelajari dan mengkaj

In [7]:
# Uji RAG normal: retrieval lalu jawaban dari GLM.
pertanyaan = "Rangkum isi halaman 6"
print(f"\nMemproses pertanyaan: '{pertanyaan}'...\n")

docs = retriever.invoke(pertanyaan)
context_str = "\n\n".join(doc.page_content for doc in docs)
messages = prompt.format_messages(context=context_str, input=pertanyaan)
response = llm.invoke(messages)

print("==================== JAWABAN AI ====================")
print(response.content)
print("====================================================")
print("\n> Bukti/Sumber Dokumen:")
for doc in docs:
    clean_content = doc.page_content[:150].replace('\n', ' ')
    print(f"  - Halaman {doc.metadata.get('page', 'Unknown')}: {clean_content}...")


Memproses pertanyaan: 'Rangkum isi halaman 6'...

==================== JAWABAN AI ====================
Isi teks ini merupakan pengantar Modul 6 yang menjelaskan bahwa perumusan kebijakan publik adalah tahap sangat krusial karena menentukan keberhasilan atau kegagalan proses implementasi, evaluasi, dan perubahan kebijakan selanjutnya. Untuk menghasilkan kebijakan yang baik, proses perumusan harus dilakukan secara optimal. Modul ini secara khusus membahas empat pokok bahasan utama, yaitu model perumusan, arena dan aktor, desain kebijakan, serta proses adopsi dan legitimasi kebijakan. Selain itu, terdapat *Gambar 9.1* yang mengilustrasikan siklus perubahan kebijakan mulai dari perumusan, pemantauan hasil, pembentukan konstituen, mobilisasi sumber daya, hingga desain organisasi. Proses perumusan ini juga memiliki kaitan erat dengan tahap sebelumnya, yakni proses identifikasi masalah dan pemasukan masalah ke dalam agenda kebijakan.

> Bukti/Sumber Dokumen:
  - Halaman 311: kepentingan;  20

In [ ]:
# Coba tes dengan pertanyaan yang kemungkinan tidak ada di dokumen,
# untuk menguji agar AI tidak berhalusinasi.
pertanyaan = "Siapa pemenang piala dunia tahun 2022?"
print(f"\nMemproses pertanyaan: '{pertanyaan}'...\n")

docs = retriever.invoke(pertanyaan)
context_str = "\n\n".join(doc.page_content for doc in docs)
messages = prompt.format_messages(context=context_str, input=pertanyaan)
response = llm.invoke(messages)

print("==================== JAWABAN AI ====================")
print(response.content)
print("====================================================")
print("\n> Bukti/Sumber Dokumen:")
for doc in docs:
    clean_content = doc.page_content[:150].replace('\n', ' ')
    print(f"  - Halaman {doc.metadata.get('page', 'Unknown')}: {clean_content}...")


Memproses pertanyaan: 'Siapa pemenang piala dunia tahun 2022?'...

==================== JAWABAN AI ====================
Saya tidak tahu jawabannya karena dokumen yang disediakan tidak mengandung informasi mengenai pemenang Piala Dunia tahun 2022.

> Bukti/Sumber Dokumen:
  - Halaman 307: masyarakat sipil dan sektor privat. Otoritas dan responsibilitas tersebar di antara  aktor-aktor yang terlibat sehingga tidak ada satu pun yang berper...
  - Halaman 40: Kunci Jawaban Tes Formatif  Tes Formatif I  1)  2)  3)  4)  5)  6)  2)  8)  9)  10)  20005565_ADPUS410_EDISI 3 1SLindb  35  A  PDOUSF RDP  Tes Formati...


: 